#### Project 2 — CFPB Triage Latency Prediction

#### Feature Engineering + Model Experiments + Model Selection

#### Objective

- Determine which intake-time feature representation and regression
formulation can predict `triage_delay_days`.

#### Frozen split

Train:- 2026-03-01 → 2026-05-27

Validation:- 2026-05-27 → 2026-06-17

Test:- 2026-06-17 → 2026-08-11

- The test set remains locked.

#### Primary metric:- MAE

#### Secondary metrics

- Median Absolute Error
- RMSE
- R²

#### Modeling principle

- Only information available at `Date received` may be used as a feature.

- No feature will be engineered from `Date sent to company` or any
downstream outcome.

In [1]:
## Imports & Paths
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import (
    Ridge,
    HuberRegressor
)

from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    median_absolute_error,
    mean_squared_error,
    r2_score
)

from scipy import sparse

PROJECT_ROOT = Path.cwd().parent.parent

PROCESSED_DIR = PROJECT_ROOT / "Data" / "processed"
REPORT_DIR = PROJECT_ROOT / "Reports" / "project2"

TRAIN_PATH = PROCESSED_DIR / "project2_train.csv"
VAL_PATH = PROCESSED_DIR / "project2_validation.csv"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
## Load Frozen Development Data
train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VAL_PATH)

train_df["Date received"] = pd.to_datetime(
    train_df["Date received"],
    utc=True
)

validation_df["Date received"] = pd.to_datetime(
    validation_df["Date received"],
    utc=True
)

y_train = train_df["triage_delay_days"]
y_validation = validation_df["triage_delay_days"]

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)

Train: (57362, 17)
Validation: (12292, 17)


#### 1. Feature Engineering

- Feature engineering will follow an evidence hierarchy.

##### Layer 1 — Time

- Information directly available from `Date received`.

#### Layer 2 — Complaint metadata

- Product, taxonomy, company, geography.

#### Layer 3 — Narrative

- Complaint text available at intake.

#### Layer 4 — Operational proxies

- Narrative length and other complexity indicators.

#### Important

- A feature is not included merely because it improves validation MAE. It must first pass the prediction-time availability test.

In [4]:
## Time Features
def add_time_features(data):
    result = data.copy()

    result["received_hour"] = result["Date received"].dt.hour
    result["received_dayofweek"] = result["Date received"].dt.dayofweek
    result["received_month"] = result["Date received"].dt.month
    result["received_day"] = result["Date received"].dt.day

    result["is_weekend"] = (
        result["received_dayofweek"] >= 5
    ).astype(int)

    return result


train_fe = add_time_features(train_df)
validation_fe = add_time_features(validation_df)

display(
    train_fe[
        [
            "received_hour",
            "received_dayofweek",
            "received_month",
            "received_day",
            "is_weekend"
        ]
    ].head()
)

,received_hour,received_dayofweek,received_month,received_day,is_weekend
0,0,6,3,1,1
1,1,6,3,1,1
2,1,6,3,1,1
3,1,6,3,1,1
4,1,6,3,1,1


In [5]:
## Basic Operational
def add_text_features(data):
    result = data.copy()

    narrative = (
        result["Consumer complaint narrative"]
        .fillna("")
        .astype(str)
    )

    result["narrative_char_count"] = narrative.str.len()
    result["narrative_word_count"] = (
        narrative.str.split().str.len()
    )

    result["narrative_exclamation_count"] = (
        narrative.str.count("!")
    )

    result["narrative_question_count"] = (
        narrative.str.count(r"\?")
    )

    return result


train_fe = add_text_features(train_fe)
validation_fe = add_text_features(validation_fe)

#### Feature Availability Configuration
#### Candidate Feature Sets

#### Experiment A — Time only

- Tests whether operational timing itself explains latency.

#### Experiment B — Time + metadata

- Adds intake metadata.

#### Experiment C — Time + metadata + text statistics

- Adds simple proxies for narrative complexity.

#### Experiment D — Time + metadata + narrative TF-IDF

- Tests whether complaint content contains predictive signal.

#### Taxonomy fields

- Product / Sub-product / Issue / Sub-issue remain conditional features.

- They should be included only if operationally available at the prediction
event.

- For this experiment we begin conservatively with Product.


In [6]:
## Define Feature Sets
time_features = [
    "received_hour",
    "received_dayofweek",
    "received_month",
    "received_day",
    "is_weekend"
]

metadata_features = [
    "Product",
    "Company",
    "State"
]

text_stat_features = [
    "narrative_char_count",
    "narrative_word_count",
    "narrative_exclamation_count",
    "narrative_question_count"
]

print("Time:", time_features)
print("Metadata:", metadata_features)
print("Text statistics:", text_stat_features)

Time: ['received_hour', 'received_dayofweek', 'received_month', 'received_day', 'is_weekend']
Metadata: ['Product', 'Company', 'State']
Text statistics: ['narrative_char_count', 'narrative_word_count', 'narrative_exclamation_count', 'narrative_question_count']


In [7]:
## Evaluation Helper
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "Median_AE": median_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "R2": r2_score(y_true, y_pred)
    }

In [8]:
## Baseline Comparision
baseline_predictions = {
    "Train Mean": np.full(
        len(validation_df),
        y_train.mean()
    ),
    "Train Median": np.full(
        len(validation_df),
        y_train.median()
    )
}

baseline_results = []

for name, predictions in baseline_predictions.items():
    baseline_results.append({
        "experiment": name,
        "target": "raw",
        **regression_metrics(
            y_validation,
            predictions
        )
    })

baseline_results = pd.DataFrame(baseline_results)

display(baseline_results)

,experiment,target,MAE,Median_AE,RMSE,R2
0,Train Mean,raw,3.034631,2.209497,5.510240,-0.043087
1,Train Median,raw,1.093132,0.005822,5.503912,-0.040692


In [9]:
## Ridge with controlled Feature Sets
def build_ridge_pipeline(numeric_features, categorical_features):
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(strategy="median")
                    )
                ]),
                numeric_features
            ),
            (
                "cat",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),
                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore"
                        )
                    )
                ]),
                categorical_features
            )
        ]
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=1.0))
    ])

In [10]:
## Run structured Feature Experiments
feature_experiments = [
    (
        "Time only",
        time_features,
        []
    ),
    (
        "Time + Product",
        time_features,
        ["Product"]
    ),
    (
        "Time + Metadata",
        time_features,
        metadata_features
    ),
    (
        "Time + Metadata + Text Statistics",
        time_features + text_stat_features,
        metadata_features
    )
]

ridge_results = []

for name, numeric_features, categorical_features in feature_experiments:

    model = build_ridge_pipeline(
        numeric_features,
        categorical_features
    )

    columns = numeric_features + categorical_features

    model.fit(
        train_fe[columns],
        y_train
    )

    predictions = model.predict(
        validation_fe[columns]
    )

    ridge_results.append({
        "experiment": name,
        "target": "raw",
        **regression_metrics(
            y_validation,
            predictions
        )
    })

ridge_results = pd.DataFrame(ridge_results)

display(
    ridge_results.sort_values("MAE")
)

,experiment,target,MAE,Median_AE,RMSE,R2
2,Time + Metadata,raw,3.056413,1.747514,5.829608,-0.167503
3,Time + Metadata + Text Statistics,raw,3.070255,1.760524,5.837962,-0.170852
1,Time + Product,raw,3.276042,2.589773,5.604170,-0.078952
0,Time only,raw,3.302129,2.389741,5.594477,-0.075223


##### 2. Narrative Feature Experiment

The narrative is available at intake and therefore can legitimately
contribute to the prediction.

We begin with TF-IDF rather than immediately using a transformer.

Reason:

- interpretable
- fast
- strong baseline
- suitable for establishing whether text contains signal
- avoids unnecessary complexity before evidence justifies it

In [11]:
## TF-IDF
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=10000,
    sublinear_tf=True
)

X_train_text = word_vectorizer.fit_transform(
    train_fe["Consumer complaint narrative"]
    .fillna("")
)

X_validation_text = word_vectorizer.transform(
    validation_fe["Consumer complaint narrative"]
    .fillna("")
)

print("Train text matrix:", X_train_text.shape)
print("Validation text matrix:", X_validation_text.shape)

Train text matrix: (57362, 10000)
Validation text matrix: (12292, 10000)


In [12]:
## Sparse Text Regression
text_model = Ridge(alpha=1.0)

text_model.fit(
    X_train_text,
    y_train
)

text_predictions = text_model.predict(
    X_validation_text
)

text_result = pd.DataFrame([
    {
        "experiment": "Narrative TF-IDF + Ridge",
        "target": "raw",
        **regression_metrics(
            y_validation,
            text_predictions
        )
    }
])

display(text_result)

,experiment,target,MAE,Median_AE,RMSE,R2
0,Narrative TF-IDF + Ridge,raw,3.75388,2.531401,6.203497,-0.322064


In [13]:
## Combined Metadata + Text
metadata_encoder = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),
            metadata_features
        ),
        (
            "num",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median")
                )
            ]),
            time_features + text_stat_features
        )
    ]
)

X_train_meta = metadata_encoder.fit_transform(
    train_fe[
        metadata_features
        + time_features
        + text_stat_features
    ]
)

X_validation_meta = metadata_encoder.transform(
    validation_fe[
        metadata_features
        + time_features
        + text_stat_features
    ]
)

X_train_combined = sparse.hstack([
    X_train_meta,
    X_train_text
]).tocsr()

X_validation_combined = sparse.hstack([
    X_validation_meta,
    X_validation_text
]).tocsr()

print("Combined train:", X_train_combined.shape)
print("Combined validation:", X_validation_combined.shape)

Combined train: (57362, 11851)
Combined validation: (12292, 11851)


In [14]:
## combined Ridge
combined_model = Ridge(alpha=1.0)

combined_model.fit(
    X_train_combined,
    y_train
)

combined_predictions = combined_model.predict(
    X_validation_combined
)

combined_result = pd.DataFrame([
    {
        "experiment": "Metadata + Time + Text + Ridge",
        "target": "raw",
        **regression_metrics(
            y_validation,
            combined_predictions
        )
    }
])

display(combined_result)

,experiment,target,MAE,Median_AE,RMSE,R2
0,Metadata + Time + Text + Ridge,raw,3.750644,2.494566,6.324245,-0.374032


##### Log-target experiment
- This is an important experiment to skewness of 6.03.
##### 3. Raw Target vs Log Target
- The raw target is extremely right-skewed.
We therefore test:
##### Model A
Predict:- `triage_delay_days`
##### Model B
Predict:- `log1p(triage_delay_days)`
- The log model's predictions are converted back to days using:- `expm1(prediction)`
##### Important
- We evaluate BOTH models on the original day scale. Otherwise the comparison would not be meaningful for the business.

In [15]:
## Train Log-Target Model
log_y_train = np.log1p(y_train)

log_model = Ridge(alpha=1.0)

log_model.fit(
    X_train_combined,
    log_y_train
)

log_predictions = log_model.predict(
    X_validation_combined
)

log_predictions_days = np.expm1(
    log_predictions
)

log_result = pd.DataFrame([
    {
        "experiment": "Metadata + Time + Text + Ridge",
        "target": "log1p",
        **regression_metrics(
            y_validation,
            log_predictions_days
        )
    }
])

display(log_result)

,experiment,target,MAE,Median_AE,RMSE,R2
0,Metadata + Time + Text + Ridge,log1p,1.413875,0.262566,5.375969,0.007129


In [16]:
### Compare All Experiments
all_results = pd.concat(
    [
        baseline_results,
        ridge_results,
        text_result,
        combined_result,
        log_result
    ],
    ignore_index=True
)

all_results = all_results.sort_values(
    "MAE"
).reset_index(drop=True)

display(all_results)

,experiment,target,MAE,Median_AE,RMSE,R2
0,Train Median,raw,1.093132,0.005822,5.503912,-0.040692
1,Metadata + Time + Text + Ridge,log1p,1.413875,0.262566,5.375969,0.007129
2,Train Mean,raw,3.034631,2.209497,5.510240,-0.043087
3,Time + Metadata,raw,3.056413,1.747514,5.829608,-0.167503
4,Time + Metadata + Text Statistics,raw,3.070255,1.760524,5.837962,-0.170852
5,Time + Product,raw,3.276042,2.589773,5.604170,-0.078952
6,Time only,raw,3.302129,2.389741,5.594477,-0.075223
7,Metadata + Time + Text + Ridge,raw,3.750644,2.494566,6.324245,-0.374032
8,Narrative TF-IDF + Ridge,raw,3.753880,2.531401,6.203497,-0.322064


In [17]:
## Tail Performence
def evaluate_by_delay_band(
    y_true,
    y_pred,
    thresholds=(1, 7, 30)
):
    result = []

    for threshold in thresholds:
        mask = y_true > threshold

        if mask.sum() == 0:
            continue

        result.append({
            "actual_delay": f">{threshold} days",
            "count": int(mask.sum()),
            "MAE": mean_absolute_error(
                y_true[mask],
                y_pred[mask]
            ),
            "Median_AE": median_absolute_error(
                y_true[mask],
                y_pred[mask]
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    y_true[mask],
                    y_pred[mask]
                )
            )
        })

    return pd.DataFrame(result)


display(
    evaluate_by_delay_band(
        y_validation,
        combined_predictions
    )
)

,actual_delay,count,MAE,Median_AE,RMSE
0,>1 days,712,14.409118,10.971766,19.193821
1,>7 days,635,15.638849,12.088414,20.179187
2,>30 days,110,36.749727,39.452755,39.275329


In [18]:
## Over Prediction/Under Prediction
residuals = y_validation - combined_predictions

error_analysis = pd.DataFrame({
    "actual": y_validation.values,
    "predicted": combined_predictions,
    "residual": residuals.values,
    "absolute_error": np.abs(residuals.values)
})

print("Mean residual:", error_analysis["residual"].mean())
print("MAE:", error_analysis["absolute_error"].mean())

print("\nLargest underpredictions:")
display(
    error_analysis
    .sort_values("residual", ascending=False)
    .head(10)
)

print("\nLargest overpredictions:")
display(
    error_analysis
    .sort_values("residual", ascending=True)
    .head(10)
)

Mean residual: -1.2508366341450887
MAE: 3.750644228405138

Largest underpredictions:


,actual,predicted,residual,absolute_error
310,91.845995,2.586973,89.259023,89.259023
3749,49.517245,-6.887885,56.405130,56.405130
4911,59.096435,3.252608,55.843827,55.843827
2321,51.170035,-2.136477,53.306512,53.306512
380,62.712801,10.035819,52.676981,52.676981
2341,50.972234,-0.908832,51.881066,51.881066
3027,50.024884,-1.642893,51.667778,51.667778
4744,49.504410,-1.653187,51.157597,51.157597
4763,47.982616,-2.795669,50.778284,50.778284
4153,49.906134,0.133736,49.772398,49.772398



Largest overpredictions:


,actual,predicted,residual,absolute_error
7042,0.009167,45.655709,-45.646543,45.646543
9733,0.008981,44.486676,-44.477694,44.477694
8609,0.008137,44.074364,-44.066227,44.066227
972,0.011308,44.066529,-44.055222,44.055222
8396,0.024826,44.066321,-44.041495,44.041495
7878,0.008912,43.367792,-43.358880,43.358880
3625,0.002604,43.192019,-43.189415,43.189415
8342,0.006238,41.597800,-41.591562,41.591562
11123,0.012465,41.323936,-41.311471,41.311471
8187,0.009097,40.393775,-40.384678,40.384678


##### 4. Model Selection

- Model selection will NOT be based on one metric alone.

##### Primary criterion:- Validation MAE.

##### Secondary criteria

- Median Absolute Error
- RMSE
- R²
- long-delay performance
- residual behavior
- operational interpretability
- model complexity

##### Important trade-off

- A model with slightly better overall MAE may still be inferior if it
systematically underpredicts the long-delay cases.

- Therefore the final candidate must balance:- `Typical-case accuracy + Tail behavior + Generalization`

In [19]:
best_by_mae = all_results.iloc[0]

print("Best validation model by MAE:")
display(best_by_mae.to_frame("value"))

Best validation model by MAE:


,value
experiment,Train Median
target,raw
MAE,1.093132
Median_AE,0.005822
RMSE,5.503912
R2,-0.040692


In [20]:
## Save Experiments Results
all_results.to_csv(
    REPORT_DIR / "project2_model_experiment_results.csv",
    index=False
)

tail_results = evaluate_by_delay_band(
    y_validation,
    combined_predictions
)

tail_results.to_csv(
    REPORT_DIR / "project2_tail_performance.csv",
    index=False
)

print("Experiment reports saved.")

Experiment reports saved.


##### Final Analysis Insights

##### 1. Temporal split

The temporal split reveals substantial process drift.

Later complaints contain dramatically fewer long-delay observations.

This is important operational information, not a reason to abandon
temporal evaluation.

##### 2. Feature engineering

Feature engineering is restricted to information available at intake.

No downstream target information is used.

##### 3. Model progression

Models are compared in increasing complexity:

Baseline
→ structured intake features
→ narrative
→ combined representation
→ raw vs log-target formulation

##### 4. Target formulation

- The raw target remains the canonical business target.

- The log1p formulation is evaluated as an alternative modeling strategy
because of the extreme right skew.

##### 5. Evaluation

MAE is the primary metric.

- Tail-specific errors are explicitly evaluated because overall MAE can
hide poor performance on long-delay complaints.

##### 6. Model selection

- The final candidate will be selected using validation evidence,
not test performance.

##### 7. Test protection

- The test set has not been used for feature engineering, model
selection, or hyperparameter tuning.